# **MRC**

In [ ]:
from transformers import (
    BertForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
    EvalPrediction,
)
from datasets import load_from_disk
import numpy as np
import torch

In [ ]:
def compute_metrics(p: EvalPrediction):
    start_preds, end_preds = p.predictions[0], p.predictions[1]
    start_labels, end_labels = p.label_ids[0], p.label_ids[1]

    start_acc = np.mean(np.argmax(start_preds, axis=1) == start_labels)
    end_acc = np.mean(np.argmax(end_preds, axis=1) == end_labels)
    return {"start_accuracy": start_acc, "end_accuracy": end_acc}

def main():
    model = BertForQuestionAnswering.from_pretrained("bert-base-uncased")
    model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

    train_dataset = load_from_disk("./data/tokenized_train")
    eval_dataset = load_from_disk("./data/tokenized_dev")

    args = TrainingArguments(
        output_dir="./outputs",
        eval_strategy="epoch",
        save_strategy="epoch",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        learning_rate=3e-5,
        weight_decay=0.01,
        logging_dir="./logs",
        logging_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        fp16=torch.cuda.is_available(),  # use float16 if available (for speed)
        report_to=[],
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=None,
        data_collator=default_data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    trainer.save_model("outputs/final_mrc_model_base")

In [ ]:
main()

# **Discourse**

In [ ]:
"""
Molweni Phase 2 — Discourse-Aware MRC (Baseline + EM/F1 Eval + Graph-ready hooks)
=================================================================================

This single-file script gives you:
  1) Data preprocessing to a canonical SQuAD-style schema (with unique IDs)
  2) Tokenization for train/dev/test with correct start/end alignment and offsets
  3) HuggingFace Trainer training loop (GPU if available, no wandb by default)
  4) Proper post-processing to text spans + EM/F1 computation on test
  5) Phase-2: A discourse-aware enhancement path that's **runnable today** via a
     lightweight Utterance Encoder + global discourse vector fusion (no graph deps)
  6) Clear TODO hooks to upgrade to a full Discourse Graph model later

Usage (typical):
  - Put Molweni MRC(withDiscourse) jsons here:
        data/Molweni/MRC(withDiscourse)/train.json
        data/Molweni/MRC(withDiscourse)/dev.json
        data/Molweni/MRC(withDiscourse)/test.json
  - Then run:
        python molweni_phase2.py --stage preprocess
        python molweni_phase2.py --stage train
        python molweni_phase2.py --stage eval

You can also run end-to-end:
        python molweni_phase2.py --stage all

Notes:
  - No internet used. Set --model_name to any local HF model.
  - Discourse-aware enhancement is opt-in with --use_discourse True.
    It computes a global discourse vector from dialogue utterances and fuses it
    into token representations before the QA head (residual add + LayerNorm).
  - This does NOT change context text or answer offsets, so no re-alignment pain.
  - Full graph modeling hooks are included as stubs (upgrade later without rewrites).
"""

import argparse
import json
import os
from pathlib import Path
from typing import Dict, List, Tuple, Any

import numpy as np
import torch
import torch.nn as nn
from datasets import Dataset, load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForQuestionAnswering,
    PreTrainedTokenizerBase,
    TrainingArguments,
    Trainer,
    default_data_collator,
)

# ----------------------------
# Config defaults (override via CLI)
# ----------------------------
DEFAULT_MODEL = "bert-base-uncased"
MAX_LEN = 512
DOC_STRIDE = 128
N_BEST = 20
MAX_ANS_LEN = 30

DATA_DIR = Path("data")
TOK_DIR = Path("data")
TOK_TRAIN = TOK_DIR / "tokenized_train"
TOK_DEV   = TOK_DIR / "tokenized_dev"
TOK_TEST  = TOK_DIR / "tokenized_test"

OUTPUT_DIR = Path("outputs")
LOG_DIR = Path("logs")

# ----------------------------
# Stage 0: Load & Canonicalize (adds "id")
# ----------------------------

def load_and_prepare(split_path: Path) -> List[Dict[str, Any]]:
    """Load Molweni json and normalize to SQuAD-style entries with unique IDs.

    Each sample fields:
        id: str
        context: str
        question: str
        answers: {"text": [str], "answer_start": [int]}
        is_impossible: bool (SQuAD v2 style)

    EXPECTED Molweni structure (simplified):
        { "data": { "dialogues": [ {"context": str, "qas": [...],
                                       "utterances": [...], "relations": [...] } ] } }
    We only need context/qas here. Utterances/relations are used later for discourse.
    """
    obj = json.load(open(split_path, "r", encoding="utf-8"))
    samples: List[Dict[str, Any]] = []
    uid = 0

    for entry in obj["data"]["dialogues"]:
        context = entry["context"]
        for q in entry["qas"]:
            question = q["question"]
            is_impossible = q.get("is_impossible", False)
            if is_impossible:
                samples.append({
                    "id": str(uid),
                    "context": context,
                    "question": question,
                    "answers": {"text": [""], "answer_start": [0]},
                    "is_impossible": True,
                })
            else:
                # Molweni typically has a single gold span per Q
                ans = q["answers"][0]
                samples.append({
                    "id": str(uid),
                    "context": context,
                    "question": question,
                    "answers": {"text": [ans["text"]], "answer_start": [ans["answer_start"]]},
                    "is_impossible": False,
                })
            uid += 1
    return samples

# ----------------------------
# Stage 1: Tokenization + alignment
# ----------------------------

def prepare_train_features(examples: Dict[str, List], tokenizer: PreTrainedTokenizerBase) -> Dict[str, Any]:
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LEN,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")  # pop once (train doesn't keep offsets)

    start_positions = []
    end_positions = []

    # default fallbacks if field missing in a batch
    is_imp_list = examples.get("is_impossible", [False] * len(examples["question"]))

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized.sequence_ids(i)
        sample_idx = sample_mapping[i]

        # choose one gold (Molweni has one; generalize anyway)
        ans_starts = examples["answers"][sample_idx]["answer_start"]
        ans_texts  = examples["answers"][sample_idx]["text"]
        impossible = is_imp_list[sample_idx]

        if impossible or not ans_starts or not ans_texts or ans_texts[0] == "":
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = ans_starts[0]
        end_char = start_char + len(ans_texts[0])

        # find context token boundaries in this feature
        token_start_idx = 0
        while token_start_idx < len(sequence_ids) and sequence_ids[token_start_idx] != 1:
            token_start_idx += 1
        token_end_idx = len(input_ids) - 1
        while token_end_idx >= 0 and sequence_ids[token_end_idx] != 1:
            token_end_idx -= 1

        # if answer not fully in this feature, point to CLS
        if token_start_idx >= len(sequence_ids) or token_end_idx < 0 or \
           offsets[token_start_idx][0] > start_char or \
           offsets[token_end_idx][1] < end_char:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        # narrow down to exact token indices
        s = token_start_idx
        e = token_end_idx
        while s <= token_end_idx and offsets[s][0] <= start_char:
            if offsets[s][1] > start_char:
                break
            s += 1
        while e >= token_start_idx and offsets[e][1] >= end_char:
            if offsets[e][0] < end_char:
                break
            e -= 1

        start_positions.append(s if s <= token_end_idx else cls_index)
        end_positions.append(e if e >= token_start_idx else cls_index)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized


def prepare_val_features(examples: Dict[str, List], tokenizer: PreTrainedTokenizerBase) -> Dict[str, Any]:
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LEN,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    tokenized["example_id"] = []

    # keep offset_mapping for post-processing; mask question side offsets
    for i in range(len(tokenized["input_ids"])):
        sample_idx = sample_mapping[i]
        tokenized["example_id"].append(examples["id"][sample_idx])

        sequence_ids = tokenized.sequence_ids(i)
        offsets = tokenized["offset_mapping"][i]
        tokenized["offset_mapping"][i] = [
            (o if sequence_ids[k] == 1 else None) for k, o in enumerate(offsets)
        ]
    return tokenized

# ----------------------------
# Stage 2: Postprocess + Metrics (EM/F1)
# ----------------------------

def postprocess_predictions(examples: Dataset, features: Dataset, preds: Tuple[np.ndarray, np.ndarray],
                            tokenizer: PreTrainedTokenizerBase) -> Dict[str, str]:
    start_logits, end_logits = preds

    # map example_id -> list of feature indices
    exid_to_featidx: Dict[str, List[int]] = {}
    for i, ex_id in enumerate(features["example_id"]):
        exid_to_featidx.setdefault(ex_id, []).append(i)

    final_preds: Dict[str, str] = {}

    for ex in examples:
        ex_id = ex["id"]
        context = ex["context"]
        feat_idxs = exid_to_featidx.get(ex_id, [])

        valid_ans = []
        min_null = None

        for fi in feat_idxs:
            s_logits = start_logits[fi]
            e_logits = end_logits[fi]
            offsets = features["offset_mapping"][fi]
            input_ids = features["input_ids"][fi]

            cls_index = input_ids.index(tokenizer.cls_token_id)
            null_score = s_logits[cls_index] + e_logits[cls_index]
            if min_null is None or null_score < min_null:
                min_null = null_score

            start_indexes = np.argsort(s_logits)[-1: -N_BEST-1: -1].tolist()
            end_indexes   = np.argsort(e_logits)[-1: -N_BEST-1: -1].tolist()

            for si in start_indexes:
                for ei in end_indexes:
                    if si >= len(offsets) or ei >= len(offsets):
                        continue
                    if offsets[si] is None or offsets[ei] is None:
                        continue
                    if ei < si:
                        continue
                    length = ei - si + 1
                    if length > MAX_ANS_LEN:
                        continue
                    start_char = offsets[si][0]
                    end_char = offsets[ei][1]
                    text = context[start_char:end_char]
                    score = s_logits[si] + e_logits[ei]
                    valid_ans.append({"text": text, "score": score})

        if valid_ans:
            best = sorted(valid_ans, key=lambda x: x["score"], reverse=True)[0]
            final_preds[ex_id] = best["text"]
        else:
            final_preds[ex_id] = ""

    return final_preds


def _normalize(s: str) -> str:
    import re
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = " ".join(s.split())
    return s


def compute_em_f1(preds: Dict[str, str], refs: Dict[str, List[str]]) -> Dict[str, float]:
    em = 0.0
    f1 = 0.0
    for qid, golds in refs.items():
        pred = preds.get(qid, "")
        pred_n = _normalize(pred)
        golds_n = [_normalize(g) for g in golds if g is not None]
        if not golds_n:
            golds_n = [""]

        exact = 1.0 if pred_n in golds_n else 0.0
        em += exact

        ptoks = pred_n.split()
        best_f = 0.0
        for g in golds_n:
            gtoks = g.split()
            if len(ptoks) == 0 and len(gtoks) == 0:
                cur = 1.0
            elif len(ptoks) == 0 or len(gtoks) == 0:
                cur = 0.0
            else:
                # multiset overlap
                common = {}
                for t in ptoks:
                    common[t] = min(ptoks.count(t), gtoks.count(t))
                inter = sum(common.values())
                prec = inter / len(ptoks) if ptoks else 0.0
                rec  = inter / len(gtoks) if gtoks else 0.0
                cur = 0.0 if (prec + rec) == 0 else 2 * prec * rec / (prec + rec)
            best_f = max(best_f, cur)
        f1 += best_f

    n = max(1, len(refs))
    return {"exact_match": 100 * em / n, "f1": 100 * f1 / n}

# ----------------------------
# Phase 2 (runnable today): Discourse-aware fusion via Utterance Encoder
# ----------------------------
class DiscourseAwareQAModel(nn.Module):
    """A wrapper around a base encoder + QA head, with optional discourse vector fusion.

    - If use_discourse=False: behaves like AutoModelForQuestionAnswering.
    - If use_discourse=True: computes a global discourse vector from dialogue utterances
      (provided in batch under key 'utter_texts') using a small frozen encoder, and fuses
      it into token hidden states before the QA head (residual add + LayerNorm).

    This avoids changing the context string or answer offsets.
    """
    def __init__(self, model_name: str, use_discourse: bool, tokenizer: PreTrainedTokenizerBase):
        super().__init__()
        self.use_discourse = use_discourse
        self.tokenizer = tokenizer

        # base encoder + QA head equivalent to AutoModelForQuestionAnswering
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.qa_outputs = nn.Linear(hidden, 2)
        self.dropout = nn.Dropout(getattr(self.encoder.config, "hidden_dropout_prob", 0.1))

        if self.use_discourse:
            # a lightweight utterance encoder (reuse same base encoder weights, frozen)
            self.utt_encoder = AutoModel.from_pretrained(model_name)
            for p in self.utt_encoder.parameters():
                p.requires_grad = False
            self.proj = nn.Linear(hidden, hidden)
            self.ln = nn.LayerNorm(hidden)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None,
                start_positions=None, end_positions=None, utter_texts=None):
        # 1) get token-level hidden states
        outputs = self.encoder(input_ids=input_ids,
                               attention_mask=attention_mask,
                               token_type_ids=token_type_ids,
                               return_dict=True)
        last_hidden = outputs.last_hidden_state  # (B, T, H)

        if self.use_discourse and utter_texts is not None:
            # compute a global discourse vector per example by encoding
            # each dialogue utterance separately, then average [CLS]
            device = last_hidden.device
            B = input_ids.size(0)
            glob_vecs = []
            for b in range(B):
                utts: List[str] = utter_texts[b]
                if not utts:
                    glob_vecs.append(torch.zeros(self.encoder.config.hidden_size, device=device))
                    continue
                cls_list = []
                for u in utts:
                    enc = self.tokenizer(u, return_tensors="pt", truncation=True, max_length=64)
                    enc = {k: v.to(device) for k, v in enc.items()}
                    with torch.no_grad():
                        out = self.utt_encoder(**enc)
                    cls_list.append(out.last_hidden_state[:, 0, :])  # (1, H)
                cls_stack = torch.cat(cls_list, dim=0)  # (U, H)
                glob = cls_stack.mean(dim=0)  # (H,)
                glob_vecs.append(glob)
            glob_vec = torch.stack(glob_vecs, dim=0)  # (B, H)
            glob_vec = self.proj(glob_vec).unsqueeze(1)  # (B, 1, H)

            # fuse via residual + LayerNorm
            last_hidden = self.ln(last_hidden + glob_vec)

        x = self.dropout(last_hidden)
        logits = self.qa_outputs(x)  # (B, T, 2)
        start_logits, end_logits = logits.split(1, dim=-1)
        start_logits = start_logits.squeeze(-1)
        end_logits = end_logits.squeeze(-1)

        total_loss = None
        if start_positions is not None and end_positions is not None:
            # clamp to length
            ignored_index = start_logits.size(1)
            start_positions = start_positions.clamp(0, ignored_index)
            end_positions = end_positions.clamp(0, ignored_index)
            loss_fct = nn.CrossEntropyLoss(ignore_index=ignored_index)
            start_loss = loss_fct(start_logits, start_positions)
            end_loss = loss_fct(end_logits, end_positions)
            total_loss = (start_loss + end_loss) / 2

        return {
            "loss": total_loss,
            "start_logits": start_logits,
            "end_logits": end_logits,
        }

# ----------------------------
# Optional: Custom data collator to pass utter_texts (for discourse fusion)
# ----------------------------
class DiscourseCollator:
    """Wrap default collator but also forwards 'utter_texts' (list[list[str]]).

    If batch examples contain 'utter_texts', we keep it in the batch; otherwise do nothing.
    """
    def __init__(self, tokenizer: PreTrainedTokenizerBase):
        self.base = default_data_collator
        self.tokenizer = tokenizer

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        # Separate string lists from tensor features
        utter_texts = None
        if "utter_texts" in features[0]:
            utter_texts = [f.get("utter_texts", []) for f in features]
            # drop from features before tensorization
            for f in features:
                f.pop("utter_texts", None)
        batch = self.base(features)
        if utter_texts is not None:
            batch["utter_texts"] = utter_texts
        return batch

# ----------------------------
# Utility: build utter_texts per example (from Molweni)
# ----------------------------

def attach_utter_texts(split_path: Path, samples: List[Dict[str, Any]]) -> List[List[str]]:
    """Given the raw Molweni split and the flattened samples (1 per Q),
    return a parallel list of utterance text lists per sample's dialogue.
    Assumes samples were generated in the same iteration order as dialogues->qas.
    """
    obj = json.load(open(split_path, "r", encoding="utf-8"))
    out: List[List[str]] = []
    cursor = 0
    for dlg in obj["data"]["dialogues"]:
        utts = [u.get("text", "") for u in dlg.get("utterances", [])]
        num_q = len(dlg.get("qas", []))
        for _ in range(num_q):
            out.append(utts)
            cursor += 1
    assert len(out) == len(samples), "Utterance list alignment mismatch"
    return out

# ----------------------------
# Train / Eval pipeline
# ----------------------------

def stage_preprocess(args):
    os.makedirs(TOK_DIR, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL)

    # train
    train_samples = load_and_prepare(DATA_DIR / "train.json")
    train_ds = Dataset.from_list(train_samples)
    train_feats = train_ds.map(lambda ex: prepare_train_features(ex, tokenizer),
                               batched=True,
                               remove_columns=train_ds.column_names)
    train_feats.save_to_disk(str(TOK_TRAIN))
    print("Saved:", TOK_TRAIN)

    # dev
    dev_samples = load_and_prepare(DATA_DIR / "dev.json")
    dev_ds = Dataset.from_list(dev_samples)
    dev_feats = dev_ds.map(lambda ex: prepare_val_features(ex, tokenizer),
                           batched=True,
                           remove_columns=dev_ds.column_names)
    dev_feats.save_to_disk(str(TOK_DEV))
    print("Saved:", TOK_DEV)

    # test
    test_samples = load_and_prepare(DATA_DIR / "test.json")
    test_ds = Dataset.from_list(test_samples)
    test_feats = test_ds.map(lambda ex: prepare_val_features(ex, tokenizer),
                             batched=True,
                             remove_columns=test_ds.column_names)
    test_feats.save_to_disk(str(TOK_TEST))
    print("Saved:", TOK_TEST)


def make_trainer(args):
    tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL)

    train_dataset = load_from_disk(str(TOK_TRAIN))
    eval_dataset  = load_from_disk(str(TOK_DEV))

    if args["use_discourse"]:
        # attach utter_texts to the *original* dev/train sample lists for collator
        # We reload raw samples to align utterances; then merge into tokenized datasets
        train_samples = load_and_prepare(DATA_DIR / "train.json")
        dev_samples   = load_and_prepare(DATA_DIR / "dev.json")
        train_utts = attach_utter_texts(DATA_DIR / "train.json", train_samples)
        dev_utts   = attach_utter_texts(DATA_DIR / "dev.json", dev_samples)

        # Add to tokenized datasets as a new column
        train_dataset = train_dataset.add_column("utter_texts", train_utts)
        eval_dataset  = eval_dataset.add_column("utter_texts", dev_utts)

        model = DiscourseAwareQAModel(DEFAULT_MODEL, use_discourse=True, tokenizer=tokenizer)
        data_collator = DiscourseCollator(tokenizer)
    else:
        model = AutoModelForQuestionAnswering.from_pretrained(DEFAULT_MODEL)
        data_collator = default_data_collator

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=args["lr"],
        per_device_train_batch_size=args["train_bs"],
        per_device_eval_batch_size=args["eval_bs"],
        num_train_epochs=args["epochs"],
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        report_to=[],  # disable wandb by default
        logging_dir=str(LOG_DIR),
        logging_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

    return trainer, tokenizer


def stage_train(args):
    trainer, _ = make_trainer(args)
    trainer.train()
    trainer.save_model(str(OUTPUT_DIR / "final_mrc_model"))
    print("Model saved to:", OUTPUT_DIR / "final_mrc_model")


def stage_eval(args):
    # load tokenized features and raw examples for refs
    tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL)
    test_feats = load_from_disk(str(TOK_TEST))

    # need raw examples for gold answers & ids
    test_samples = load_and_prepare(DATA_DIR / "test.json")
    test_examples = Dataset.from_list(test_samples)

    # rebuild trainer (loads best model)
    trainer, _ = make_trainer(args)
    trainer.model.eval()

    # Swap eval_dataset to test features
    trainer.eval_dataset = test_feats

    raw = trainer.predict(test_feats)
    preds = postprocess_predictions(test_examples, test_feats, raw.predictions, tokenizer)

    refs = {ex["id"]: ex["answers"]["text"] for ex in test_examples}
    metrics = compute_em_f1(preds, refs)
    print("Test EM/F1:", metrics)


# ----------------------------
# CLI
# ----------------------------

def parse_args():
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", type=str, default="all",
                    choices=["preprocess", "train", "eval", "all"],
                    help="Which pipeline step to run")
    ap.add_argument("--model_name", type=str, default=DEFAULT_MODEL)
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--train_bs", type=int, default=8)
    ap.add_argument("--eval_bs", type=int, default=8)
    ap.add_argument("--lr", type=float, default=3e-5)
    ap.add_argument("--use_discourse", type=lambda x: str(x).lower() in {"true","1","yes"}, default=False,
                    help="Enable discourse-aware fusion (utterance encoder)")
    return ap.parse_args()


def main():
    # args = parse_args()
    args = {
        "stage": "all",
        "model_name": DEFAULT_MODEL,
        "epochs": 3,
        "train_bs": 8,
        "eval_bs": 8,
        "lr": 3e-5,
        "use_discourse": False,
    }
    # if args["stage"] in ("preprocess", "all"):
    print("[Stage] Preprocess")
    stage_preprocess(args)

    # if args["stage"] in ("train", "all"):
    print("[Stage] Train")
    stage_train(args)

    # if args["stage"] in ("eval", "all"):
    print("[Stage] Eval")
    stage_eval(args)


# if __name__ == "__main__":
main()


In [ ]:
# molweni_deepseq.py
# Full end-to-end DeepSequential Discourse QA (Molweni) — single-file runnable script.

import argparse
import json
import os
from pathlib import Path
from collections import defaultdict
from typing import List, Dict, Any, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
)

# PyG
from torch_geometric.nn import GATConv

# -------------------
# Config
# -------------------
DEFAULT_MODEL = "bert-base-uncased"
DATA_DIR = Path("data")
TOKENIZED_DIR = Path("data/tokenized")
TOKENIZED_DIR.mkdir(parents=True, exist_ok=True)
TOK_TRAIN = TOKENIZED_DIR / "train"
TOK_DEV   = TOKENIZED_DIR / "dev"
TOK_TEST  = TOKENIZED_DIR / "test"

MAX_LEN = 512
DOC_STRIDE = 128
N_BEST = 20
MAX_ANS_LEN = 30

# -------------------
# Helpers: load and canonicalize
# -------------------
def load_and_prepare(split_path: Path) -> List[Dict[str, Any]]:
    obj = json.load(open(split_path, "r", encoding="utf-8"))
    samples = []
    uid = 0
    for dlg in obj["data"]["dialogues"]:
        context = dlg["context"]
        # keep utterances and relations for Phase 2
        utterances = dlg.get("utterances", [])
        relations = dlg.get("relations", [])
        for q in dlg["qas"]:
            is_imp = q.get("is_impossible", False)
            if is_imp:
                samples.append({
                    "id": str(uid),
                    "context": context,
                    "question": q["question"],
                    "answers": {"text": [""], "answer_start": [0]},
                    "is_impossible": True,
                    "utterances": utterances,
                    "relations": relations
                })
            else:
                ans = q["answers"][0]
                samples.append({
                    "id": str(uid),
                    "context": context,
                    "question": q["question"],
                    "answers": {"text": [ans["text"]], "answer_start": [ans["answer_start"]]},
                    "is_impossible": False,
                    "utterances": utterances,
                    "relations": relations
                })
            uid += 1
    return samples

# -------------------
# Tokenization + feature alignment
# -------------------
def prepare_train_features(examples: Dict[str, List], tokenizer: AutoTokenizer):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LEN,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")  # popped for train

    start_positions, end_positions = [], []
    is_imp_list = examples.get("is_impossible", [False]*len(examples["question"]))

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized.sequence_ids(i)
        sample_idx = sample_mapping[i]

        ans_starts = examples["answers"][sample_idx]["answer_start"]
        ans_texts  = examples["answers"][sample_idx]["text"]
        impossible = is_imp_list[sample_idx]

        if impossible or not ans_starts or not ans_texts or ans_texts[0] == "":
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = ans_starts[0]
        end_char = start_char + len(ans_texts[0])

        token_start_idx = 0
        while token_start_idx < len(sequence_ids) and sequence_ids[token_start_idx] != 1:
            token_start_idx += 1
        token_end_idx = len(input_ids) - 1
        while token_end_idx >= 0 and sequence_ids[token_end_idx] != 1:
            token_end_idx -= 1

        if token_start_idx >= len(sequence_ids) or token_end_idx < 0 \
           or offsets[token_start_idx][0] > start_char or offsets[token_end_idx][1] < end_char:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        s = token_start_idx
        while s <= token_end_idx and offsets[s][1] <= start_char:
            s += 1
        e = token_end_idx
        while e >= token_start_idx and offsets[e][0] >= end_char:
            e -= 1

        start_positions.append(s if s <= token_end_idx else cls_index)
        end_positions.append(e if e >= token_start_idx else cls_index)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized

def prepare_val_features(examples: Dict[str, List], tokenizer: AutoTokenizer):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LEN,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    tokenized["example_id"] = []
    # keep offset_mapping but set to None for question tokens
    for i in range(len(tokenized["input_ids"])):
        sample_idx = sample_mapping[i]
        tokenized["example_id"].append(examples["id"][sample_idx])

        sequence_ids = tokenized.sequence_ids(i)
        offsets = tokenized["offset_mapping"][i]
        tokenized["offset_mapping"][i] = [
            (o if sequence_ids[k] == 1 else None) for k, o in enumerate(offsets)
        ]
    # Also add utterances and relations aligned to feature chunks so collator can pick them
    # We will map feature -> sample idx via overflow mapping; for simplicity, we add lists synced by sample
    return tokenized

# -------------------
# Postprocessing: logits -> predictions -> EM/F1
# -------------------
def postprocess_predictions(examples: Dataset, features: Dataset, preds: Tuple[np.ndarray, np.ndarray], tokenizer: AutoTokenizer):
    start_logits, end_logits = preds
    feat_map = defaultdict(list)
    for i, exid in enumerate(features["example_id"]):
        feat_map[exid].append(i)

    final_preds = {}
    for ex in examples:
        ex_id = ex["id"]
        context = ex["context"]
        fids = feat_map.get(ex_id, [])
        valid = []
        for fid in fids:
            s_logits = start_logits[fid]
            e_logits = end_logits[fid]
            offsets = features["offset_mapping"][fid]
            input_ids = features["input_ids"][fid]
            cls_index = input_ids.index(tokenizer.cls_token_id)

            start_idxs = np.argsort(s_logits)[-1: -N_BEST-1: -1].tolist()
            end_idxs = np.argsort(e_logits)[-1: -N_BEST-1: -1].tolist()

            for si in start_idxs:
                for ei in end_idxs:
                    if si >= len(offsets) or ei >= len(offsets):
                        continue
                    if offsets[si] is None or offsets[ei] is None:
                        continue
                    if ei < si or (ei - si + 1) > MAX_ANS_LEN:
                        continue
                    s_char = offsets[si][0]
                    e_char = offsets[ei][1]
                    txt = context[s_char:e_char]
                    score = s_logits[si] + e_logits[ei]
                    valid.append({"text": txt, "score": score})
        if valid:
            best = sorted(valid, key=lambda x: x["score"], reverse=True)[0]
            final_preds[ex_id] = best["text"]
        else:
            final_preds[ex_id] = ""
    return final_preds

def normalize_answer(s: str) -> str:
    import re
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = " ".join(s.split())
    return s

def compute_em_f1(preds: Dict[str, str], refs: Dict[str, List[str]]):
    em = 0.0
    f1 = 0.0
    for qid, golds in refs.items():
        pred = normalize_answer(preds.get(qid, ""))
        golds_norm = [normalize_answer(g) for g in golds] if golds else [""]
        exact = 1.0 if pred in golds_norm else 0.0
        em += exact
        ptoks = pred.split()
        best_f = 0.0
        for gold in golds_norm:
            gtoks = gold.split()
            if not ptoks and not gtoks:
                cur = 1.0
            elif not ptoks or not gtoks:
                cur = 0.0
            else:
                common = 0
                from collections import Counter
                pc = Counter(ptoks); gc = Counter(gtoks)
                for k in pc:
                    common += min(pc[k], gc.get(k, 0))
                prec = common / len(ptoks) if ptoks else 0.0
                rec = common / len(gtoks) if gtoks else 0.0
                cur = 0.0 if (prec + rec) == 0 else 2 * prec * rec / (prec + rec)
            best_f = max(best_f, cur)
        f1 += best_f
    n = max(1, len(refs))
    return {"exact_match": 100 * em / n, "f1": 100 * f1 / n}

# -------------------
# DeepSequential graph block + model
# -------------------
class DeepSequential(nn.Module):
    def __init__(self, hidden_size, num_layers=2, heads=4):
        super().__init__()
        self.layers = nn.ModuleList([
            GATConv(hidden_size, hidden_size // heads, heads=heads, concat=True)
            for _ in range(num_layers)
        ])
    def forward(self, x, edge_index):
        # x: (U, H); edge_index: (2, E)
        h = x
        for layer in self.layers:
            h = layer(h, edge_index)
            h = F.relu(h)
        return h

class DiscourseGraphQAModel(nn.Module):
    def __init__(self, model_name: str, num_relations: int, tokenizer: AutoTokenizer):
        super().__init__()
        self.tokenizer = tokenizer
        self.encoder = AutoModel.from_pretrained(model_name)
        self.hidden = self.encoder.config.hidden_size
        self.qa_outputs = nn.Linear(self.hidden, 2)
        self.dropout = nn.Dropout(0.1)

        # utterance encoder (frozen) to get utterance node initials
        self.utt_encoder = AutoModel.from_pretrained(model_name)
        for p in self.utt_encoder.parameters():
            p.requires_grad = False

        self.graph = DeepSequential(self.hidden, num_layers=2, heads=4)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None,
                start_positions=None, end_positions=None,
                utter_texts=None, relations=None):
        # Base token encoding
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
                               token_type_ids=token_type_ids, return_dict=True)
        seq_out = outputs.last_hidden_state  # (B, T, H)

        # If discourse info provided, compute per-example graph and fuse
        device = seq_out.device
        if utter_texts is not None and relations is not None:
            # utter_texts: list[list[str]] (len B)
            glob_vectors = []
            for b_idx in range(len(utter_texts)):
                utts = utter_texts[b_idx]  # list[str]
                rels = relations[b_idx]     # list of {"x":..,"y":..,"type":..} or similar
                if not utts:
                    glob_vectors.append(torch.zeros(self.hidden, device=device))
                    continue
                # encode utterances (use tokenizer and utt_encoder)
                cls_list = []
                for u in utts:
                    enc = self.tokenizer(u, return_tensors="pt", truncation=True, max_length=64)
                    enc = {k: v.to(device) for k, v in enc.items()}
                    with torch.no_grad():
                        out = self.utt_encoder(**enc)
                    cls_list.append(out.last_hidden_state[:, 0, :])  # (1, H)
                utt_feats = torch.cat(cls_list, dim=0)  # (U, H)

                # build edge_index from rels
                if rels:
                    src = [r["x"] for r in rels]
                    tgt = [r["y"] for r in rels]
                    edge_index = torch.tensor([src, tgt], dtype=torch.long, device=device)
                else:
                    edge_index = torch.zeros((2, 0), dtype=torch.long, device=device)

                # run graph module
                if edge_index.size(1) == 0:
                    updated = utt_feats  # no edges -> identity
                else:
                    updated = self.graph(utt_feats, edge_index)  # (U, H)

                glob = updated.mean(dim=0)  # (H,)
                glob_vectors.append(glob)
            # stack -> (B, H) -> unsqueeze to (B, 1, H) and broadcast
            glob_stack = torch.stack(glob_vectors, dim=0).unsqueeze(1)  # (B,1,H)
            seq_out = seq_out + glob_stack  # residual fuse

        x = self.dropout(seq_out)
        logits = self.qa_outputs(x)  # (B, T, 2)
        start_logits, end_logits = logits.split(1, dim=-1)
        start_logits = start_logits.squeeze(-1)
        end_logits = end_logits.squeeze(-1)

        loss = None
        if start_positions is not None and end_positions is not None:
            ignored_index = start_logits.size(1)
            start_positions = start_positions.clamp(0, ignored_index)
            end_positions = end_positions.clamp(0, ignored_index)
            loss_fct = nn.CrossEntropyLoss(ignore_index=ignored_index)
            start_loss = loss_fct(start_logits, start_positions)
            end_loss = loss_fct(end_logits, end_positions)
            loss = (start_loss + end_loss) / 2

        return {"loss": loss, "start_logits": start_logits, "end_logits": end_logits}


# -------------------
# Collator: keep utter_texts and relations lists and let default collator tensorize the rest
# -------------------
class GraphCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.base = default_data_collator

    def __call__(self, features: List[Dict[str, Any]]):
        # extract utter_texts & relations lists
        utter_texts = [f.pop("utterances", []) for f in features]
        relations = [f.pop("relations", []) for f in features]
        batch = self.base(features)  # tensorize standard fields
        batch["utter_texts"] = utter_texts
        batch["relations"] = relations
        return batch

# -------------------
# Pipeline actions
# -------------------
def stage_preprocess(args):
    tokenizer = AutoTokenizer.from_pretrained(args["model_name"])

    # Train
    train_samples = load_and_prepare(DATA_DIR / "train.json")
    train_ds = Dataset.from_list(train_samples)
    print("Tokenizing train...")
    train_feats = train_ds.map(lambda ex: prepare_train_features(ex, tokenizer),
                               batched=True, remove_columns=train_ds.column_names)
    train_feats.save_to_disk(str(TOK_TRAIN))
    print("Saved tokenized train ->", TOK_TRAIN)

    # Dev
    dev_samples = load_and_prepare(DATA_DIR / "dev.json")
    dev_ds = Dataset.from_list(dev_samples)
    print("Tokenizing dev...")
    dev_feats = dev_ds.map(lambda ex: prepare_val_features(ex, tokenizer),
                           batched=True, remove_columns=dev_ds.column_names)
    dev_feats.save_to_disk(str(TOK_DEV))
    print("Saved tokenized dev ->", TOK_DEV)

    # Test
    test_samples = load_and_prepare(DATA_DIR / "test.json")
    test_ds = Dataset.from_list(test_samples)
    print("Tokenizing test...")
    test_feats = test_ds.map(lambda ex: prepare_val_features(ex, tokenizer),
                             batched=True, remove_columns=test_ds.column_names)
    test_feats.save_to_disk(str(TOK_TEST))
    print("Saved tokenized test ->", TOK_TEST)

def make_trainer(args):
    tokenizer = AutoTokenizer.from_pretrained(args["model_name"])
    # load tokenized datasets
    train_ds = load_from_disk(str(TOK_TRAIN))
    dev_ds = load_from_disk(str(TOK_DEV))

    # model
    model = DiscourseGraphQAModel(args["model_name"], num_relations=10, tokenizer=tokenizer)

    training_args = TrainingArguments(
        output_dir="./outputs",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=args["lr"],
        per_device_train_batch_size=args["train_bs"],
        per_device_eval_batch_size=args["eval_bs"],
        num_train_epochs=args["epochs"],
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        report_to=[]
    )

    collator = GraphCollator(tokenizer)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=dev_ds,
        tokenizer=tokenizer,
        data_collator=collator
    )
    return trainer, tokenizer

def stage_train(args):
    trainer, _ = make_trainer(args)
    trainer.train()
    trainer.save_model("outputs/final_deepseq_model")
    print("Saved final model.")

def stage_eval(args):
    trainer, tokenizer = make_trainer(args)
    # load test features + raw examples
    test_feats = load_from_disk(str(TOK_TEST))
    test_raw = load_and_prepare(DATA_DIR / "test.json")
    test_examples = Dataset.from_list(test_raw)

    trainer.model.eval()
    trainer.eval_dataset = test_feats
    raw = trainer.predict(test_feats)
    preds = postprocess_predictions(test_examples, test_feats, raw.predictions, tokenizer)
    refs = {ex["id"]: ex["answers"]["text"] for ex in test_examples}
    metrics = compute_em_f1(preds, refs)
    print("Test EM/F1:", metrics)

# -------------------
# CLI
# -------------------
def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--stage", choices=["preprocess","train","eval","all"], default="all")
    p.add_argument("--model_name", default=DEFAULT_MODEL)
    p.add_argument("--epochs", type=int, default=3)
    p.add_argument("--train_bs", type=int, default=4)
    p.add_argument("--eval_bs", type=int, default=8)
    p.add_argument("--lr", type=float, default=3e-5)
    return p.parse_args()

def main():
    # args = parse_args()
    args = {
        "stage": "all",
        "model_name": DEFAULT_MODEL,
        "epochs": 3,
        "train_bs": 4,
        "eval_bs": 8,
        "lr": 3e-5,
    }
    # if args["stage"] in ("preprocess", "all"):
    print("[Stage] Preprocess")
    stage_preprocess(args)

    # if args["stage"] in ("train", "all"):
    print("[Stage] Train")
    stage_train(args)

    # if args["stage"] in ("eval", "all"):
    print("[Stage] Eval")
    stage_eval(args)

# if __name__ == "__main__":
main()
